In [1]:
import torch
import torch.nn as nn

In [2]:
class MotorModel_1(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(4 , 64)
        self.l2 = nn.Linear(64 , 128)
        self.l3 = nn.Linear(128 , 64)
        self.l4 = nn.Linear(64 , 6)
        self.r = nn.ReLU()
        self.d = nn.Dropout(0.15)
    def forward(self , x):
        return self.l4(self.d(self.r(self.l3(self.d(self.r(self.l2(self.d(self.r(self.l1(x))))))))))


In [3]:
model = MotorModel_1()
model.load_state_dict(torch.load("saved_models/m1.pth"))
model.eval()

MotorModel_1(
  (l1): Linear(in_features=4, out_features=64, bias=True)
  (l2): Linear(in_features=64, out_features=128, bias=True)
  (l3): Linear(in_features=128, out_features=64, bias=True)
  (l4): Linear(in_features=64, out_features=6, bias=True)
  (r): ReLU()
  (d): Dropout(p=0.15, inplace=False)
)

In [4]:
import numpy as np

In [5]:
import joblib

In [9]:
scaler = joblib.load("saved_models/scaler.pkl")

In [18]:
import warnings
warnings.filterwarnings("ignore")

In [19]:
def run():
    
    torque = float(input("Enter Torque (Nm)  : "))
    flux = float(input("Enter Flux (Wb)    : "))
    angle = float(input("Enter Angle (deg)  : "))
    sin_theta = np.sin(np.radians(angle))
    cos_theta = np.cos(np.radians(angle))
    data = np.array([[torque, flux, sin_theta, cos_theta]])
    scaled_data = scaler.transform(data)
    input_tensor = torch.tensor(scaled_data, dtype=torch.float32)
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.sigmoid(output)
        prediction = (probabilities > 0.5).int()
    labels = [
        "Pulse_A_Upper",
        "Pulse_B_Upper",
        "Pulse_C_Upper",
        "Pulse_A_Lower",
        "Pulse_B_Lower",
        "Pulse_C_Lower"
    ]

    print("\n========== PREDICTED SWITCH STATES ==========\n")

    for i, value in enumerate(prediction[0]):
        if value == 1:
            print(f"✅ {labels[i]}  --->  ON")
        else:
            print(f"❌ {labels[i]}  --->  OFF")


In [20]:
run()

Enter Torque (Nm)  :  100
Enter Flux (Wb)    :  1.32
Enter Angle (deg)  :  224



========== PREDICTED SWITCH STATES ==========

❌ Pulse_A_Upper  --->  OFF
✅ Pulse_B_Upper  --->  ON
❌ Pulse_C_Upper  --->  OFF
✅ Pulse_A_Lower  --->  ON
✅ Pulse_B_Lower  --->  ON
❌ Pulse_C_Lower  --->  OFF
